In [ ]:
import random
import numpy as np
import os

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(42)


# Stacking  XGBoost+ CatBoost+ Ridge+ LGBM

* This notebook stacks the XGBoost and LGBM and Ridge and CatBoost models.

* The reference notes for data processing are below.

https://www.kaggle.com/code/mdshahbazalam/lgbm-multioutputregressor

# If this note is useful, Please VOTE!

In [ ]:
import os

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import xgboost as xgb

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression

In [ ]:
import pandas as pd
train=pd.read_csv('input/feedback-prize-english-language-learning/train.csv')
display(train.head(2))

In [ ]:
print('No. of rows in train datasets',train.shape[0])

In [ ]:
def data_cleaner(text):
    text = text.strip()
    text = re.sub(r'\n', '', text)
    text = text.lower()
    return text

In [ ]:
train['full_text']=train['full_text'].apply(data_cleaner)

In [ ]:
import nltk
from tqdm import tqdm
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
def generate_sentiment_scores(data):
    sid = SentimentIntensityAnalyzer()
    neg=[]
    pos=[]
    neu=[]
    comp=[]
    for sentence in tqdm(data['full_text'].values): 
        sentence_sentiment_score = sid.polarity_scores(sentence)
        comp.append(sentence_sentiment_score['compound'])
        neg.append(sentence_sentiment_score['neg'])
        pos.append(sentence_sentiment_score['pos'])
        neu.append(sentence_sentiment_score['neu'])
    return comp,neg,pos,neu
train['compound'],train['negative'],train['positive'],train['neutral']=generate_sentiment_scores(train)

In [ ]:
train['com_len']=train['full_text'].apply(lambda x:len(x.split()))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train['full_text'])

In [ ]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_com=trans.fit_transform(train['compound'].values.reshape(-1,1))

In [ ]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_neg=trans.fit_transform(train['negative'].values.reshape(-1,1))

In [ ]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_pos=trans.fit_transform(train['positive'].values.reshape(-1,1))

In [ ]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_neu=trans.fit_transform(train['neutral'].values.reshape(-1,1))

In [ ]:
%%time
from sklearn.preprocessing import Normalizer
trans = Normalizer()
X_train_len=trans.fit_transform(train['com_len'].values.reshape(-1,1))

In [ ]:
%%time
from scipy.sparse import hstack
train_s=hstack((X_train,X_train_com,X_train_neg,X_train_pos,X_train_neu,X_train_len))

In [ ]:
y=train[['cohesion','syntax','vocabulary','phraseology','grammar','conventions']]

In [ ]:
print(train_s.shape,y.shape)

In [ ]:
params_lgb = {
    "n_estimators": 1000,
    "verbose": -1,
    "random_state": 42
}

In [ ]:
y_train=train[['cohesion','syntax','vocabulary','phraseology','grammar','conventions']]

In [ ]:
model = MultiOutputRegressor(LGBMRegressor(**params_lgb))
model.fit(train_s, y_train)

In [ ]:
param = {'learning_rate': 0.3, 
          'depth': 12, 
          'l2_leaf_reg': 4, 
          'loss_function': 'MultiRMSE', 
          'eval_metric': 'MultiRMSE', 
          'task_type': 'CPU', 
          'iterations': 20,
          'od_type': 'Iter', 
          'boosting_type': 'Plain', 
          'bootstrap_type': 'Bayesian', 
          'allow_const_label': True, 
          'random_state': 42,
          'allow_writing_files': False
         }

In [ ]:
model2 = CatBoostRegressor(**param)
model2.fit(train_s, y_train)

In [ ]:
model3 = Ridge(copy_X=False, random_state=42)
model3.fit(train_s, y_train)

In [ ]:
xgb_estimator = xgb.XGBRegressor(
        n_estimators=500, random_state=42, 
        objective='reg:squarederror')

# create MultiOutputClassifier instance with XGBoost model inside
model4 = MultiOutputRegressor(xgb_estimator, n_jobs=2)
# model4 = XGBClassifier(early_stopping_rounds=10)
model4.fit(train_s, y_train)

In [ ]:
first_pred_1 = model.predict(train_s)
first_pred_2 = model2.predict(train_s)
first_pred_3 = model3.predict(train_s)
first_pred_4 = model4.predict(train_s)

stack_pred = np.column_stack((first_pred_1,first_pred_2,first_pred_3,first_pred_4))

In [ ]:
params = {'learning_rate': 0.3, 
          'depth': 12, 
          'l2_leaf_reg': 4, 
          'loss_function': 'MultiRMSE', 
          'eval_metric': 'MultiRMSE', 
          'task_type': 'CPU', 
          'iterations': 20,
          'od_type': 'Iter', 
          'boosting_type': 'Plain', 
          'bootstrap_type': 'Bayesian', 
          'allow_const_label': True, 
          'random_state': 42,
          'allow_writing_files': False
         }


In [ ]:
# メタモデルの作成
# meta_model = LinearRegression()
meta_model =  CatBoostRegressor(**params)
# meta_model = MultiOutputRegressor(LGBMRegressor(**params_lgb))

meta_model.fit(stack_pred, y_train)

In [ ]:
import joblib

# Save models
joblib.dump(model, 'output/model1_lgbm.pkl')
model2.save_model("output/model2_catboost.cbm")
joblib.dump(model3, 'output/model3_ridge.pkl')
joblib.dump(model4, 'output/model4_xgb.pkl')
meta_model.save_model("output/meta_model_catboost.cbm")

# Save preprocessors
joblib.dump(vectorizer, 'output/vectorizer.pkl')
joblib.dump(trans, 'output/normalizer.pkl')